# Lesson 9.3 · XGBoost in practice

Practical features that make XGBoost the reference model on tabular data: it handles **missing values** without imputation, it accepts **monotonic constraints**, it provides per-example contributions (`pred_contribs`) and it has a very fast implementation (`tree_method="hist"`).

**What you will learn**

- How XGBoost treats NaN and what changes compared with imputation.
- What the different importances (`gain`, `weight`, `cover`) and the per-example contributions mean.
- How XGBoost chooses a split (gradient, hessian, $\lambda$, $\gamma$) and where the hyperparameters come in.
- Monotonic constraints and imbalanced classes.
- How it compares with scikit-learn's `GradientBoostingClassifier` and `HistGradientBoostingClassifier`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(12000, n_features=20, n_informative=8, class_sep=0.8, flip_y=0.05, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

## Native missing values

At every split XGBoost learns **which way to send the NaN** (left or right) by choosing the direction that reduces the loss the most: missingness becomes information, like the missingness indicator of module 4 but without having to build it. You do not need an imputer (exercise B2).

## Per-example contributions

`predict(..., pred_contribs=True)` (on the `Booster` object) returns, for each example, the contribution of every feature to the prediction (on the log-odds scale) and a constant term (`bias`): the **sum of the contributions is the raw prediction**. It is a local explanation in the style of SHAP computed exactly on the trees.

In [ ]:
model = xgb.XGBClassifier(n_estimators=120, learning_rate=0.1, max_depth=3, eval_metric="logloss", n_jobs=-1, random_state=0).fit(X_tr, y_tr)
booster = model.get_booster()
contribs = booster.predict(xgb.DMatrix(X_te[:3]), pred_contribs=True)            # (examples, features + bias)
margin = booster.predict(xgb.DMatrix(X_te[:3]), output_margin=True)
print("shape of the contributions:", contribs.shape, "| sum of the contributions = raw margin:", np.allclose(contribs.sum(1), margin, atol=1e-5))
print("the three largest contributions for the first example (column index):", np.argsort(-np.abs(contribs[0, :-1]))[:3], np.round(contribs[0, np.argsort(-np.abs(contribs[0, :-1]))[:3]], 3))

## The importances: `weight`, `gain`, `cover`

`booster.get_score(importance_type=...)` counts three different things for each feature: `weight` = how many times it appears in a split, `gain` = average gain of its splits, `cover` = how many examples (in terms of hessian) go through its splits. They are three answers to three questions, and they do not coincide.

In [ ]:
table = pd.DataFrame({kind: pd.Series(booster.get_score(importance_type=kind)) for kind in ["weight", "gain", "cover"]}).fillna(0)
(table / table.sum()).sort_values("gain", ascending=False).head(6).round(3)

`gain` **concentrates** the importance: the first feature (`f0`) is worth 22% of the gain but only 11% of the splits; at the opposite extreme `f16` appears in 7% of the splits and is worth 5% of the gain. `weight` counts the cuts, not how useful they are, and it rewards continuous features with many distinct values. To decide which features are really needed, the permutation on validation data remains the reference (exercise D2).

## How XGBoost chooses a split

For a generic loss XGBoost approximates the error with its second-order expansion: for each example it computes the **gradient** $g_i$ and the **hessian** $h_i$ of the loss. In a leaf with sums $G = \sum g_i$ and $H = \sum h_i$ the optimal value is $w^* = -G/(H + \lambda)$, and a left/right split improves the loss by

$$\text{gain} = \tfrac12\left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma .$$

All the hyperparameters of lesson 9.2 are in here: $\lambda$ (`reg_lambda`) shrinks the leaf values, $\gamma$ (`gamma`) is the minimum gain to split, `min_child_weight` is a minimum for $H$. In exercise A2 you rebuild the Newton step by hand (leaf value $\sum r/\sum p(1-p)$).

## Monotonic constraints and imbalanced classes

- **Monotonic constraints:** `monotone_constraints={"income": 1, "debt": -1}` forces the prediction to be nondecreasing in `income` and nonincreasing in `debt`. In credit, rents or insurance it is often a domain requirement, and it costs little if the constraint is consistent with the data. Prefer the dictionary by name over the positional tuple (exercise E3).
- **Imbalanced classes:** `scale_pos_weight` weights the positive class. It rebalances the weights and therefore **distorts the probabilities**: if you need probabilities or a decision with costs, it is safer not to reweight and to choose the threshold (modules 4 and 10).

## Three implementations

- `GradientBoostingClassifier` (scikit-learn): the simplest, very slow on large data.
- `HistGradientBoostingClassifier` (scikit-learn): uses feature histograms and supports NaN; much faster.
- `XGBClassifier` with `tree_method="hist"`: histograms, parallelism, explicit regularization, monotonic constraints; the fastest and the richest in features.

In [ ]:
for name, m in {"GradientBoosting (sklearn)": GradientBoostingClassifier(n_estimators=60, max_depth=3, random_state=0),
                "HistGradientBoosting (sklearn)": HistGradientBoostingClassifier(max_iter=60, max_depth=3, random_state=0),
                "XGBoost hist": xgb.XGBClassifier(n_estimators=60, max_depth=3, tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=0)}.items():
    t0 = time.time(); m.fit(X_tr, y_tr); tf = time.time() - t0
    print(f"{name:32s} fit {tf:5.1f} s | accuracy {m.score(X_te, y_te):.3f}")

> **Common mistake.** Using the `weight` importance (number of times a feature is used in a split) as "importance": it rewards continuous features with many thresholds, like the impurity of a tree. `gain` (average gain) and the per-example contributions are more informative; for reliable judgments the permutation on validation data remains the choice.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>When do you prefer a random forest to XGBoost?</summary>

When you want a robust model with few hyperparameters and little sensitivity to how they are chosen, with free OOB estimates and easy parallel training; with very noisy data (boosting chases the wrong labels: exercise C2). XGBoost usually wins in accuracy if you tune it carefully.

</details>

<details><summary>Does XGBoost extrapolate?</summary>

No: it sums trees, so outside the training range it is flat like any tree-based model. For trends you need a linear model as a base (`booster="gblinear"` or a hybrid) or to transform the target (differences).

</details>